In [ ]:
import torch

print(f"torch version: {torch.__version__}")
print(f"torch cuda version: {torch.version.cuda}")
device = torch.device(torch.cuda.current_device() if torch.cuda.is_available() else "cpu")
print(f"torch device: {device}")
print(f"device name: {torch.cuda.get_device_name(device) if torch.cuda.is_available() else 'cpu'}")

torch version: 2.6.0+cu126
torch cuda version: 12.6
torch device: cuda:0
device name: NVIDIA GeForce RTX 4090 Laptop GPU


In [ ]:
import json
import logging
import os
import re
from enum import EnumType

import datasets
import pandas as pd
import tqdm
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from trl import SFTTrainer

EXIT_FAILURE = 1
EXIT_SUCCESS = 0


logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)


class MissionType(EnumType):
    WA = 1
    SA = 2
    SE = 3


MISC_DATA_PATH = os.path.join(os.path.pardir, "MISC")
# MISC_DATA_PATH = os.path.join(os.path.curdir, "drive/MyDrive/COMP6713/MISC")
JOB_DATA_PATH = os.path.join(MISC_DATA_PATH, "job_data_files")

print(f"MISC_DATA_PATH: {MISC_DATA_PATH}")
print(f"JOB_DATA_PATH: {JOB_DATA_PATH}")


class JobDataPath(EnumType):
    WA_DEV = "work_arrangements_development_set.csv"
    WA_TEST = "work_arrangements_test_set.csv"
    SA_DEV = "salary_labelled_development_set.csv"
    SA_TEST = "salary_labelled_test_set.csv"
    SE_DEV = "seniority_labelled_development_set.csv"
    SE_TEST = "seniority_labelled_test_set.csv"


class JobData:
    def __init__(self, mission_type: MissionType):
        func_join_path = lambda x: os.path.join(JOB_DATA_PATH, x)
        match mission_type:
            case MissionType.WA:
                self.df_dev = pd.read_csv(func_join_path(JobDataPath.WA_DEV))
                self.df_test = pd.read_csv(func_join_path(JobDataPath.WA_TEST))
            case MissionType.SA:
                self.df_dev = pd.read_csv(func_join_path(JobDataPath.SA_DEV))
                self.df_test = pd.read_csv(func_join_path(JobDataPath.SA_TEST))
            case MissionType.SE:
                self.df_dev = pd.read_csv(func_join_path(JobDataPath.SE_DEV))
                self.df_test = pd.read_csv(func_join_path(JobDataPath.SE_TEST))
            case _:
                self.df_dev = None
                self.df_test = None
                raise ValueError(f"Invalid mission type: {mission_type}")

    pass


pass

MISC_DATA_PATH: ..\MISC
JOB_DATA_PATH: ..\MISC\job_data_files


In [ ]:
class MistralData:
    def __init__(self, mission_type: MissionType):
        func_join_path = lambda x: os.path.join(JOB_DATA_PATH, x)
        match mission_type:
            case MissionType.WA:
                # Load the data
                self.df_dev = pd.read_csv(func_join_path(JobDataPath.WA_DEV))
                self.df_test = pd.read_csv(func_join_path(JobDataPath.WA_TEST))
                # Create train dataset
                train_jsonl_list = self.__output_prompt_jsonl_WA(self.df_dev)
                train_jsonl_path = os.path.join(
                    MISC_DATA_PATH, "mistral", "work_arrangements_train.jsonl"
                )
                self.__write_jsonl(train_jsonl_path, train_jsonl_list)
                self.train_dataset = load_dataset("json", data_files=train_jsonl_path)
                # Create test dataset
                test_jsonl_list = self.__output_prompt_jsonl_WA(self.df_test)
                test_jsonl_path = os.path.join(
                    MISC_DATA_PATH, "mistral", "work_arrangements_test.jsonl"
                )
                self.__write_jsonl(test_jsonl_path, test_jsonl_list)
                self.test_dataset = load_dataset("json", data_files=test_jsonl_path)
            case MissionType.SA:
                # Load the data
                self.df_dev = pd.read_csv(func_join_path(JobDataPath.SA_DEV))
                self.df_test = pd.read_csv(func_join_path(JobDataPath.SA_TEST))
                # Create train dataset
                train_jsonl_list = self.__output_prompt_jsonl_SA(self.df_dev)
                train_jsonl_path = os.path.join(
                    MISC_DATA_PATH, "mistral", "salary_train.jsonl"
                )
                self.__write_jsonl(train_jsonl_path, train_jsonl_list)
                self.train_dataset = load_dataset("json", data_files=train_jsonl_path)
                # Create test dataset
                test_jsonl_list = self.__output_prompt_jsonl_SA(self.df_test)
                test_jsonl_path = os.path.join(
                    MISC_DATA_PATH, "mistral", "salary_test.jsonl"
                )
                self.__write_jsonl(test_jsonl_path, test_jsonl_list)
                self.test_dataset = load_dataset("json", data_files=test_jsonl_path)
            case MissionType.SE:
                # Load the data
                self.df_dev = pd.read_csv(func_join_path(JobDataPath.SE_DEV))
                self.df_test = pd.read_csv(func_join_path(JobDataPath.SE_TEST))
                # Create train dataset
                train_jsonl_list = self.__output_prompt_jsonl_SE(self.df_dev)
                train_jsonl_path = os.path.join(
                    MISC_DATA_PATH, "mistral", "seniority_train.jsonl"
                )
                self.__write_jsonl(train_jsonl_path, train_jsonl_list)
                self.train_dataset = load_dataset("json", data_files=train_jsonl_path)
                # Create test dataset
                test_jsonl_list = self.__output_prompt_jsonl_SE(self.df_test)
                test_jsonl_path = os.path.join(
                    MISC_DATA_PATH, "mistral", "seniority_test.jsonl"
                )
                self.__write_jsonl(test_jsonl_path, test_jsonl_list)
                self.test_dataset = load_dataset("json", data_files=test_jsonl_path)
            case _:
                self.df_dev = None
                self.df_test = None
                raise ValueError(f"Invalid mission type: {mission_type}")
        pass

    def __output_prompt_jsonl_SA(self, df: pd.DataFrame) -> list:
        logger.info(f"Creating jsonl for {df.keys()} rows")
        jsonl_list = []
        for _, row in tqdm.tqdm(df.iterrows(), total=len(df), desc="Creating jsonl"):
            jsonl_list.append(
                {
                    "instruction": (
                        "You are a helpful assistant."
                        "Your task is to classify the salary level of the job advertisement. "
                        "The format will be [NUMBER]-[NUMBER]-[CURRENCY SYMBOL]-[HOURLY|DAILY|MONTHLY|YEARLY]."
                    ),
                    "input": (  # job_title job_ad_details nation_short_desc salary_additional_text
                        f"The job title is: {row['job_title']}."
                        f"Further details: {row['job_ad_details']}."
                        f"Country Codes: {row['nation_short_desc']}."
                        f"Salary information: {row['salary_additional_text']}."
                    ),
                    "output": f"Answer: {row['y_true']}",
                }
            )
        return jsonl_list

    def __output_prompt_jsonl_SE(self, df: pd.DataFrame) -> list:
        logger.info(f"Creating jsonl for {df.keys()} rows")
        jsonl_list = []
        for _, row in tqdm.tqdm(df.iterrows(), total=len(df), desc="Creating jsonl"):
            jsonl_list.append(
                {
                    "instruction": (
                        "You are a helpful assistant."
                        "Your task is to classify the seniority level of the job advertisement."
                        "The format will be [SENIORITY LEVEL]."
                    ),
                    "input": (
                        f"The job title is: {row['job_title']}."
                        f"To summarize the job: {row['job_summary']}."
                        f"Further details: {row['job_ad_details']}"
                        f"The classification is: {row['classification_name']} / {row['subclassification_name']}."
                    ),
                    "output": f"Answer: {row['y_true']}",
                }
            )
        return jsonl_list

    def __output_prompt_jsonl_WA(self, df: pd.DataFrame) -> list:
        logger.info(f"Creating jsonl for {df.keys()} rows")
        jsonl_list = []
        for _, row in tqdm.tqdm(df.iterrows(), total=len(df), desc="Creating jsonl"):
            jsonl_list.append(
                {
                    "instruction": (
                        "You are a helpful assistant."
                        "Your task is to classify which type of work arrangements the job advertisement belongs to."
                        "There are three types: ['OnSite', 'Remote', 'Hybrid']."
                    ),
                    "input": row["job_ad"],
                    "output": f"Answer: {row['y_true']}",
                }
            )
        return jsonl_list

    def __write_jsonl(self, jsonl_path: str, data: list) -> None:
        # if file exists, remove it
        if os.path.exists(jsonl_path):
            os.remove(jsonl_path)
        # create file and write data
        with open(jsonl_path, "w", encoding="utf-8") as f:
            for item in data:
                f.write(json.dumps(item, ensure_ascii=False) + "\n")

    def format_prompt(self, example) -> str:
        return f"<s>[INST] {example['instruction']} {example['input']} [/INST] {example['output']} </s>"

    def get_map_data(self, dataset: datasets.Dataset) -> datasets.Dataset:
        return dataset.map(
            lambda example: {"text": self.format_prompt(example)},
            remove_columns=["instruction", "input", "output"],
        )

    pass


pass

In [ ]:
class MistralConfig:
    def __init__(self, mission_type: MissionType) -> None:
        self.mission_type = mission_type

        self.bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
        self.lora_config = LoraConfig(
            r=8,
            lora_alpha=16,
            lora_dropout=0.1,
            bias="none",
            task_type="CAUSAL_LM",
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj"],
        )

    pass


class MistralModel:
    def __init__(self, MistralConfig: MistralConfig, MistralData: MistralData) -> None:
        self.model_id = "mistralai/Mistral-7B-Instruct-v0.3"

        self.model_config = MistralConfig

        self.device = torch.device(
            torch.cuda.current_device() if torch.cuda.is_available() else "cpu"
        )

        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_id,
            quantization_config=MistralConfig.bnb_config,
            torch_dtype=torch.bfloat16,
            device_map="auto",
        )

        self.model.config.use_cache = False
        self.model.config.pretraining_tp = 1

        self.tokenizer = AutoTokenizer.from_pretrained(self.model_id)
        self.tokenizer.pad_token = self.tokenizer.eos_token

        self.model = prepare_model_for_kbit_training(self.model)
        self.model = get_peft_model(self.model, MistralConfig.lora_config)
        self.model.generation_config.pad_token_id = self.tokenizer.pad_token_id
        self.model.gradient_checkpointing_enable()

        self.fine_tuing_data = MistralData.get_map_data(MistralData.train_dataset)

        self.trainer = None  # type: SFTTrainer | None

        self.y_pred = None  # type: list | None

        logger.info("Model and tokenizer loaded successfully.")

    def __build_train_arguments(self) -> TrainingArguments:
        return TrainingArguments(
            output_dir="./results",
            num_train_epochs=1,
            per_device_train_batch_size=2,
            gradient_accumulation_steps=4,
            optim="paged_adamw_8bit",
            save_steps=5,
            logging_steps=1,
            learning_rate=2e-4,
            weight_decay=0.001,
            fp16=False,
            bf16=False,
            max_grad_norm=0.3,
            max_steps=-1,
            warmup_ratio=0.03,
            group_by_length=True,
            lr_scheduler_type="linear",
            report_to="wandb",
            seed=42,
        )

    def train(self) -> None:
        trainer_args = self.__build_train_arguments()
        self.trainer = SFTTrainer(
            model=self.model,
            train_dataset=self.fine_tuing_data["train"],
            args=trainer_args,
            peft_config=self.model_config.lora_config,
        )
        self.trainer.train()
        self.trainer.save_state()
        logger.info("Training completed successfully.")

    def save(self) -> None:
        if self.trainer is None:
            raise ValueError(
                "Trainer is not initialized. Please train the model first."
            )
        save_model_path = "./mistral-7b-lora"
        match self.model_config.mission_type:
            case MissionType.WA:
                save_model_path = "./mistral-7b-lora-WA"
            case MissionType.SA:
                save_model_path = "./mistral-7b-lora-SA"
            case MissionType.SE:
                save_model_path = "./mistral-7b-lora-SE"
            case _:
                raise ValueError(f"Invalid mission type: {MistralConfig.mission_type}")
        self.trainer.save_model(save_model_path)
        self.tokenizer.save_pretrained(save_model_path)
        self.model.save_pretrained(save_model_path)
        logger.info("Model saved successfully.")

    def __find_answer(self, text: str) -> str:
        regex = r"Answer: (.*)"
        result = re.search(regex, text)
        if result:
            return result.group(1).strip()
        else:
            return "None"

    def predict(self, MistralData: MistralData) -> list:
        if self.trainer is None:
            raise ValueError(
                "Trainer is not initialized. Please train the model first."
            )
        self.y_pred = []
        test_data = MistralData.test_dataset["train"]
        logger.info("Predicting...")
        for i in tqdm.tqdm(range(len(test_data))):
            input_text = test_data["input"][i]
            instruction_text = test_data["instruction"][i]
            format_input = f"<s>[INST] {instruction_text} {input_text} [/INST]"
            input_ids = self.tokenizer(format_input, return_tensors="pt").input_ids.to(
                self.device
            )
            attention_mask = self.tokenizer(
                format_input, return_tensors="pt"
            ).attention_mask.to(self.device)
            self.model.gradient_checkpointing_enable()
            self.model.generation_config.pad_token_id = self.tokenizer.pad_token_id
            output = self.model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                num_return_sequences=1,
            )
            self.model.gradient_checkpointing_enable()
            # print(sequences[0]["generated_text"])
            output_text = self.tokenizer.decode(output[0], skip_special_tokens=True)
            # print(f"Output: {output_text}")
            a = self.__find_answer(output_text)
            # print(f"Answer: {a}")
            self.y_pred.append(a)
        return self.y_pred

    def report(self, MistralData: MistralData) -> None:
        if self.trainer is None:
            raise ValueError(
                "Trainer is not initialized. Please train the model first."
            )
        if self.y_pred is None:
            raise ValueError(
                "No predictions made. Please run the predict method first."
            )

        y_test = MistralData.df_test["y_true"].tolist()
        y_pred = self.y_pred

        # Calculate accuracy
        accuracy = accuracy_score(y_test, y_pred, zero_division=0)
        logger.info(f"Accuracy: {accuracy:.4f}")

        # Calculate precision
        precision = precision_score(y_test, y_pred, average="weighted", zero_division=0)
        logger.info(f"Precision: {precision:.4f}")

        # Calculate recall
        recall = recall_score(y_test, y_pred, average="weighted", zero_division=0)
        logger.info(f"Recall: {recall:.4f}")

        # Calculate F1 score
        f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)
        logger.info(f"F1 Score: {f1:.4f}")

        # Generate classification report
        report = classification_report(y_test, y_pred, zero_division=0)

        logger.info(f"Classification Report: \n{report}")

        report_dict = classification_report(y_test, y_pred, output_dict=True, zero_division=0)

        # Convert to DataFrame for better readability
        report_df = pd.DataFrame(report_dict).transpose()

        report_file_name_template = (
            "classification_report_{mission_type}_{model_version}.csv"
        )
        match self.model_config.mission_type:
            case MissionType.WA:
                report_file_name = report_file_name_template.format(
                    mission_type="work_arrangements", model_version="mistral-7b-lora"
                )
            case MissionType.SA:
                report_file_name = report_file_name_template.format(
                    mission_type="salary", model_version="mistral-7b-lora"
                )
            case MissionType.SE:
                report_file_name = report_file_name_template.format(
                    mission_type="seniority", model_version="mistral-7b-lora"
                )
            case _:
                raise ValueError(
                    f"Invalid mission type: {self.model_config.mission_type}"
                )

        report_df.to_csv(report_file_name, index=False)
        logger.info(f"Classification report saved to {report_file_name}")
        pass

    pass

In [ ]:
def run_task(mission_type: MissionType) -> None:
    Data = None
    Config = None
    match mission_type:
        case MissionType.WA:
            logger.info("Running WA task...")
            Data = MistralData(MissionType.WA)
            Config = MistralConfig(MissionType.WA)
        case MissionType.SA:
            logger.info("Running SA task...")
            Data = MistralData(MissionType.SA)
            Config = MistralConfig(MissionType.SA)
        case MissionType.SE:
            logger.info("Running SE task...")
            Data = MistralData(MissionType.SE)
            Config = MistralConfig(MissionType.SE)
        case _:
            raise ValueError(f"Invalid mission type: {mission_type}")
    Model = MistralModel(Config, Data)

    logger.info("Training the model...")

    Model.train()

    logger.info("Model training completed.")

    Model.save()

    logger.info("Model saved successfully.")

    Model.predict(Data)

    logger.info("Model prediction completed.")

    Model.report(Data)

    logger.info("Model report completed.")
    logger.info("All tasks completed successfully.")

In [ ]:
run_task(MissionType.WA)

2025-04-09 18:38:51 - INFO - Running WA task...
2025-04-09 18:38:51 - INFO - Creating jsonl for Index(['id', 'job_ad', 'y_true'], dtype='object') rows
Creating jsonl: 100%|██████████| 99/99 [00:00<00:00, 49485.89it/s]


Generating train split: 0 examples [00:00, ? examples/s]

2025-04-09 18:38:52 - INFO - Creating jsonl for Index(['id', 'job_ad', 'y_true'], dtype='object') rows
Creating jsonl: 100%|██████████| 99/99 [00:00<00:00, 58948.91it/s]


Generating train split: 0 examples [00:00, ? examples/s]

2025-04-09 18:38:53 - INFO - We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


...

Tokenizing train dataset:   0%|          | 0/99 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/99 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: mingyuancui (mingyuancui-unsw) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
1,2.403600
2,2.624500
3,2.419200
4,2.050100
5,2.099900
6,2.222100
7,1.967800
8,1.975000
9,1.807300
10,1.870700


2025-04-09 18:43:11 - INFO - Training completed successfully.
2025-04-09 18:43:11 - INFO - Model training completed.
2025-04-09 18:43:14 - INFO - Model saved successfully.
2025-04-09 18:43:14 - INFO - Model saved successfully.
2025-04-09 18:43:14 - INFO - Predicting...
  0%|          | 0/99 [00:00<?, ?it/s]`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
c:\Users\cmy20\scoop\apps\python\current\Lib\site-packages\torch\utils\checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
100%|██████████| 99/99 [02:18<00:00,  1.40s/it]
2025-04-09 18:45:33 - INFO - Model prediction completed.
2025-04-09 18:45:33 - INFO - Accuracy: 0.8485
2025-04-09 18:45:33 - INFO - Precision: 0.8485
2025-04-09 18:45:33 - INFO - Recall: 0.8485
2025-04-09 18:45:33 - INFO - F1 Score: 0.8485
2025-04-09 18:45:33 - INFO - Classification Report: 
              precision    recall  f1-score   support

      Hybrid       0.

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj"],
)


se_merge_model = AutoModelForCausalLM.from_pretrained(
    "./mistral-7b-lora-SE",
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

se_tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.3")
se_tokenizer.pad_token = se_tokenizer.eos_token

In [ ]:
FuncFindAnswer = lambda text: re.search(r"Answer: (.*)", text).group(1).strip() if re.search(r"Answer: (.*)", text) else "None"

def predict(Model, Tokenizer, MistralData: MistralData) -> list:
    y_pred = []
    test_data = MistralData.test_dataset["train"]
    logger.info("Predicting...")
    for i in tqdm.tqdm(range(len(test_data))):
        input_text = test_data["input"][i]
        instruction_text = test_data["instruction"][i]
        format_input = f"<s>[INST] {instruction_text} {input_text} [/INST]"
        input_ids = Tokenizer(format_input, return_tensors="pt").input_ids.to(
            device
        )
        attention_mask = Tokenizer(
            format_input, return_tensors="pt"
        ).attention_mask.to(device)
        Model.gradient_checkpointing_enable()
        Model.generation_config.pad_token_id = Tokenizer.pad_token_id
        output = Model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            num_return_sequences=1,
        )
        Model.gradient_checkpointing_enable()
        # print(sequences[0]["generated_text"])
        output_text = Tokenizer.decode(output[0], skip_special_tokens=True)
        # print(f"Output: {output_text}")
        a = FuncFindAnswer(output_text)
        # print(f"Answer: {a}")
        y_pred.append(a)
    return y_pred

y_pred = predict(se_merge_model, se_tokenizer, MistralData(MissionType.SE))
y_test = MistralData(MissionType.SE).df_test["y_true"].tolist()

print(f"y_test: {y_test}")
print(f"y_pred: {y_pred}")

2025-04-09 20:51:10 - INFO - Creating jsonl for Index(['job_id', 'job_title', 'job_summary', 'job_ad_details',
       'classification_name', 'subclassification_name', 'y_true'],
      dtype='object') rows
Creating jsonl: 100%|██████████| 2752/2752 [00:00<00:00, 73813.44it/s]


Generating train split: 0 examples [00:00, ? examples/s]

2025-04-09 20:51:12 - INFO - Creating jsonl for Index(['job_id', 'job_title', 'job_summary', 'job_ad_details',
       'classification_name', 'subclassification_name', 'y_true'],
      dtype='object') rows
Creating jsonl: 100%|██████████| 689/689 [00:00<00:00, 52932.97it/s]


Generating train split: 0 examples [00:00, ? examples/s]

2025-04-09 20:51:12 - INFO - Predicting...
100%|██████████| 689/689 [06:48<00:00,  1.69it/s]
2025-04-09 20:58:00 - INFO - Creating jsonl for Index(['job_id', 'job_title', 'job_summary', 'job_ad_details',
       'classification_name', 'subclassification_name', 'y_true'],
      dtype='object') rows
Creating jsonl: 100%|██████████| 2752/2752 [00:00<00:00, 44660.14it/s]


Generating train split: 0 examples [00:00, ? examples/s]

2025-04-09 20:58:04 - INFO - Creating jsonl for Index(['job_id', 'job_title', 'job_summary', 'job_ad_details',
       'classification_name', 'subclassification_name', 'y_true'],
      dtype='object') rows
Creating jsonl: 100%|██████████| 689/689 [00:00<00:00, 49211.99it/s]


Generating train split: 0 examples [00:00, ? examples/s]

y_test: ['senior', 'experienced', 'entry level', 'senior', 'intermediate', 'experienced', 'experienced', 'entry level', 'entry level', 'experienced', 'senior', 'experienced', 'experienced', 'senior', 'trainee', 'experienced', 'experienced', 'intermediate', 'apprentice', 'entry level', 'entry level', 'entry level', 'experienced', 'experienced', 'experienced', 'experienced', 'graduate', 'senior', 'senior', 'intermediate', 'head', 'senior', 'supervisor', 'lead', 'experienced', 'entry level', 'experienced', 'trainee', 'entry level', 'intermediate', 'experienced', 'entry level', 'experienced', 'lead', 'senior', 'entry level', 'experienced', 'assistant', 'intermediate', 'intermediate', 'experienced assistant', 'assistant', 'intermediate', 'lead', 'entry level', 'experienced', 'senior', 'experienced', 'experienced', 'assistant', 'intermediate', 'entry level', 'senior', 'intermediate', 'experienced', 'experienced', 'experienced', 'junior', 'senior', 'intermediate', 'senior', 'experienced', 'le

In [ ]:
# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: \t {accuracy:.8f}")
# Calculate precision

precision = precision_score(y_test, y_pred, average="weighted", zero_division=0)
print(f"Precision: \t {precision:.8f}")

# Calculate recall
recall = recall_score(y_test, y_pred, average="weighted", zero_division=0)

print(f"Recall: \t {recall:.8f}")

# Calculate F1 score
f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)
print(f"F1 Score: \t {f1:.8f}")

# Generate classification report
report = classification_report(y_test, y_pred, zero_division=0)
print(f"Classification Report: \n{report}")

Accuracy: 	 0.70682148
Precision: 	 0.70394435
Recall: 	 0.70682148
F1 Score: 	 0.70128947
Classification Report: 
                             precision    recall  f1-score   support

        1st year apprentice       0.00      0.00      0.00         1
    Chief Executive Officer       0.00      0.00      0.00         0
                   advanced       1.00      1.00      1.00         1
                 apprentice       0.75      1.00      0.86         3
                  assistant       0.77      0.86      0.81        28
        assistant principal       0.00      0.00      0.00         0
                  associate       0.83      0.83      0.83         6
         associate director       0.67      1.00      0.80         2
         associate-director       0.00      0.00      0.00         1
                      cadet       1.00      0.50      0.67         2
                      chief       1.00      0.67      0.80         3
                 consultant       0.00      0.00      0.

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj"],
)


sa_merge_model = AutoModelForCausalLM.from_pretrained(
    "./mistral-7b-lora-SA",
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

sa_tokenizer = AutoTokenizer.from_pretrained("./mistral-7b-lora-SA")
sa_tokenizer.pad_token = sa_tokenizer.eos_token

2025-04-09 23:58:50 - INFO - We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [14]:
FuncFindAnswer = lambda text: re.search(r"Answer: (.*)", text).group(1).strip() if re.search(r"Answer: (.*)", text) else "None"

def predict(Model, Tokenizer, MistralData: MistralData) -> list:
    y_pred = []
    test_data = MistralData.test_dataset["train"]
    logger.info("Predicting...")
    for i in tqdm.tqdm(range(len(test_data))):
        input_text = test_data["input"][i]
        instruction_text = test_data["instruction"][i]
        format_input = f"<s>[INST] {instruction_text} {input_text} [/INST]"
        input_ids = Tokenizer(format_input, return_tensors="pt").input_ids.to(
            device
        )
        attention_mask = Tokenizer(
            format_input, return_tensors="pt"
        ).attention_mask.to(device)
        Model.gradient_checkpointing_enable()
        Model.generation_config.pad_token_id = Tokenizer.pad_token_id
        output = Model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            num_return_sequences=1,
        )
        Model.gradient_checkpointing_enable()
        # print(sequences[0]["generated_text"])
        output_text = Tokenizer.decode(output[0], skip_special_tokens=True)
        # print(f"Output: {output_text}")
        a = FuncFindAnswer(output_text)
        # print(f"Answer: {a}")
        y_pred.append(a)
    return y_pred

y_pred = predict(sa_merge_model, sa_tokenizer, MistralData(MissionType.SA))
y_test = MistralData(MissionType.SA).df_test["y_true"].tolist()

print(f"y_test: {y_test}")
print(f"y_pred: {y_pred}")

2025-04-09 23:59:16 - INFO - Creating jsonl for Index(['job_id', 'job_title', 'job_ad_details', 'nation_short_desc',
       'salary_additional_text', 'y_true'],
      dtype='object') rows
Creating jsonl: 100%|██████████| 2267/2267 [00:00<00:00, 56846.84it/s]


Generating train split: 0 examples [00:00, ? examples/s]

2025-04-09 23:59:17 - INFO - Creating jsonl for Index(['job_id', 'job_title', 'job_ad_details', 'nation_short_desc',
       'salary_additional_text', 'y_true'],
      dtype='object') rows
Creating jsonl: 100%|██████████| 567/567 [00:00<00:00, 56487.27it/s]


Generating train split: 0 examples [00:00, ? examples/s]

2025-04-09 23:59:17 - INFO - Predicting...
100%|██████████| 567/567 [12:22<00:00,  1.31s/it]
2025-04-10 00:11:39 - INFO - Creating jsonl for Index(['job_id', 'job_title', 'job_ad_details', 'nation_short_desc',
       'salary_additional_text', 'y_true'],
      dtype='object') rows
Creating jsonl: 100%|██████████| 2267/2267 [00:00<00:00, 45544.40it/s]


Generating train split: 0 examples [00:00, ? examples/s]

2025-04-10 00:11:40 - INFO - Creating jsonl for Index(['job_id', 'job_title', 'job_ad_details', 'nation_short_desc',
       'salary_additional_text', 'y_true'],
      dtype='object') rows
Creating jsonl: 100%|██████████| 567/567 [00:00<00:00, 54192.20it/s]


Generating train split: 0 examples [00:00, ? examples/s]

y_test: ['1500-1800-MYR-MONTHLY', '60-60-HKD-HOURLY', '0-0-None-None', '0-0-None-None', '0-0-None-None', '21-21-NZD-HOURLY', '0-0-None-None', '0-0-None-None', '32-32-AUD-HOURLY', '2000-3000-MYR-MONTHLY', '3000-4000-MYR-MONTHLY', '0-0-None-None', '0-0-None-None', '80-90-HKD-HOURLY', '142642-156491-AUD-ANNUAL', '0-0-None-None', '29-29-AUD-HOURLY', '1500-2500-MYR-MONTHLY', '66028-68086-AUD-ANNUAL', '0-0-None-None', '3000-4000-SGD-MONTHLY', '0-0-None-None', '0-0-None-None', '73-85-HKD-HOURLY', '5000-8000-MYR-MONTHLY', '30-30-NZD-HOURLY', '0-0-None-None', '500-1250-SGD-WEEKLY', '1500-1500-MYR-MONTHLY', '750-1125-SGD-WEEKLY', '0-0-None-None', '0-0-None-None', '0-0-None-None', '1500-1500-MYR-MONTHLY', '2000-2500-SGD-MONTHLY', '58008-58008-AUD-ANNUAL', '23-29-NZD-HOURLY', '3000-6000-MYR-MONTHLY', '31-31-NZD-HOURLY', '100-100-HKD-HOURLY', '500-667-PHP-DAILY', '107-150-SGD-DAILY', '1500-1500-MYR-MONTHLY', '20-25-AUD-HOURLY', '0-0-None-None', '0-0-None-None', '2700-3000-SGD-MONTHLY', '0-0-None-No

In [15]:
# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: \t {accuracy:.8f}")
# Calculate precision

precision = precision_score(y_test, y_pred, average="weighted", zero_division=0)
print(f"Precision: \t {precision:.8f}")

# Calculate recall
recall = recall_score(y_test, y_pred, average="weighted", zero_division=0)

print(f"Recall: \t {recall:.8f}")

# Calculate F1 score
f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)
print(f"F1 Score: \t {f1:.8f}")

# Generate classification report
report = classification_report(y_test, y_pred, zero_division=0)
print(f"Classification Report: \n{report}")

Accuracy: 	 0.85361552
Precision: 	 0.84672669
Recall: 	 0.85361552
F1 Score: 	 0.84766150
Classification Report: 
                           precision    recall  f1-score   support

            0-0-None-None       0.97      0.99      0.98       238
         10-10-NZD-HOURLY       0.00      0.00      0.00         0
         10-10-SGD-HOURLY       1.00      1.00      1.00         1
         10-40-NZD-HOURLY       0.00      0.00      0.00         1
       100-100-HKD-HOURLY       1.00      0.67      0.80         3
       100-120-HKD-HOURLY       0.67      1.00      0.80         2
       100-250-THB-HOURLY       1.00      1.00      1.00         2
     1000-1000-THB-HOURLY       1.00      1.00      1.00         1
       100000-100000-THB-       0.00      0.00      0.00         0
100000-100000-THB-MONTHLY       0.00      0.00      0.00         1
       100000-130000-AUD-       0.00      0.00      0.00         0
       102051-125151-AUD-       0.00      0.00      0.00         0
 102051-12515